# Inbound Auth

AgentCore Identity를 사용하면 AgentCore Runtime의 에이전트나 도구를 호출하는 사용자 및 애플리케이션의 인바운드 액세스(Inbound Auth)를 검증하거나 AgentCore Gateway 대상에 대한 액세스를 검증할 수 있습니다. 또한 에이전트가 외부 서비스 또는 Gateway 대상에 안전하게 액세스하도록 Outbound Auth를 제공합니다. 기존 IdP(예: Amazon Cognito)와 통합하면서, 에이전트가 독립적으로 또는 사용자를 대신해(OAuth 사용) 작업할 때 권한 경계를 적용합니다.

Inbound Auth는 AgentCore Runtime, AgentCore Gateway 또는 기타 환경에 호스팅된 에이전트나 도구를 호출하려는 요청자를 검증합니다. Inbound Auth는 IAM(SigV4 자격 증명) 또는 OAuth 권한 부여와 함께 작동합니다.

기본적으로 Amazon Bedrock AgentCore는 IAM 자격 증명을 사용하므로 에이전트에 대한 사용자 요청은 사용자의 IAM 자격 증명으로 인증됩니다. OAuth를 사용하는 경우 AgentCore Runtime 리소스 또는 AgentCore Gateway 엔드포인트를 구성할 때 다음 항목을 지정해야 합니다.

- OAuth discovery server URL: OpenID Connect 검색 URL의 `^.+/\.well-known/openid-configuration$` 패턴과 일치해야 하는 문자열

- Allowed audiences: JWT 토큰에 허용되는 audience 목록

- Allowed clients: 허용되는 client 식별자 목록

AgentCore CLI를 사용하는 경우 **configure** 명령으로 AgentCore Runtime의 권한 부여 유형과 OAuth discovery server를 지정할 수 있습니다. `CreateAgentRuntime` 작업이나 Amazon Bedrock AgentCore 콘솔을 사용할 수도 있습니다. Gateway를 생성할 때는 `CreateGateway` 작업 또는 콘솔을 사용합니다.

사용자가 에이전트를 사용하려면 먼저 클라이언트 애플리케이션에서 OAuth authorizer로 인증해야 합니다. 클라이언트는 bearer token을 받은 뒤 호출 요청에 담아 에이전트로 전달합니다. 에이전트는 토큰을 수신하면 authorization server를 통해 검증한 후 액세스를 허용합니다.


## 개요

이 튜토리얼에서는 01-AgentCore-runtime에서 배포한 에이전트를 수정하고 Cognito를 IdP로 사용하는 Inbound Auth를 구성합니다. 사용자 한 명과 app client가 포함된 Cognito User Pool을 설정하고, 해당 Cognito User Pool을 이용한 Inbound Auth로 기존 에이전트를 Amazon Bedrock AgentCore Runtime에 호스팅하는 방법을 알아봅니다. 

### 튜토리얼 아키텍처

<div style="text-align:center">
    <img src="images/inbound_auth_cognito.png" width="90%"/>
</div>

### 튜토리얼 세부 정보


| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| 에이전트 유형       | 단일                                                                             |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | AgentCore Runtime에 에이전트 호스팅, Strands Agent 및 Amazon Bedrock 모델 사용  |
| 튜토리얼 분야       | 산업 공통                                                                        |
| 예제 난이도         | 쉬움                                                                             |
| Inbound Auth        | Cognito                                                                          |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                                      |



### 튜토리얼 주요 기능

* Amazon Cognito 기반 Inbound Auth를 적용해 Amazon Bedrock AgentCore Runtime에 에이전트 호스팅
* Amazon Bedrock 모델 사용
* Strands Agents 사용


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Strands Agents
* 실행 중인 Docker

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## 인증을 위한 Amazon Cognito 설정

App client와 테스트 사용자 한 명이 포함된 Cognito User Pool을 프로비저닝하겠습니다. 배포된 MCP server에 액세스할 수 있도록 Amazon Cognito에서 JWT 토큰을 발급합니다. 이를 위해 `utils` 스크립트의 `setup_cognito_user_pool` 도우미 함수를 사용합니다.

참고: Cognito `access_token`의 유효 시간은 2시간입니다. 만료된 경우 `reauthenticate_user` 메서드로 새 `access_token`을 발급할 수 있습니다.

In [ ]:
import sys
import os

# 현재 Notebook의 디렉터리 확인
current_dir = os.path.dirname(os.path.abspath("__file__" if "__file__" in globals() else "."))

utils_dir = os.path.join(current_dir, "..")
utils_dir = os.path.abspath(utils_dir)

# sys.path에 추가
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import setup_cognito_user_pool, reauthenticate_user

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")

## AgentCore Runtime 배포를 위한 에이전트 준비

### Amazon Bedrock 모델을 사용하는 Strands Agents
01-AgentCore-runtime 튜토리얼에서 생성한 Strands Agent를 가져와 Amazon Cognito를 IdP로 사용하는 Inbound Auth를 구성하겠습니다.

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # calculator 도구 가져오기
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

# 사용자 지정 도구 생성
@tool
def weather():
    """ Get weather """ # 예제용 구현
    return "sunny"


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    페이로드로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## AgentCore Runtime에 에이전트 배포

`CreateAgentRuntime` 작업은 컨테이너 이미지, 환경 변수, 암호화 설정 등 폭넓은 구성 옵션을 지원합니다. 프로토콜 설정(HTTP, MCP)과 권한 부여 메커니즘도 구성하여 클라이언트가 에이전트와 통신하는 방식을 제어할 수 있습니다. 

**참고:** 운영 환경에서는 코드를 컨테이너로 패키징하고 CI/CD 파이프라인과 IaC를 사용해 ECR에 푸시하는 것이 좋습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK로 아티팩트를 간편하게 패키징하고 AgentCore Runtime에 배포합니다.

### AgentCore Runtime 배포 구성

이제 starter toolkit을 사용해 엔트리포인트, 앞에서 생성한 실행 역할, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 시작 시 Amazon ECR 리포지토리를 자동 생성하도록 starter toolkit도 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

**중요** - 앞 단계에서 확인한 Cognito Discovery URL과 Cognito App client ID로 업데이트하세요.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
region

discovery_url = cognito_config.get("discovery_url")

client_id = cognito_config.get("client_id")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_agent_inbound_identity",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
        }
    },
)
response

## AgentCore 구성 검토

In [ ]:
!cat .bedrock_agentcore.yaml

### AgentCore Runtime에 에이전트 시작

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime에 시작하겠습니다. 이 과정에서 Amazon ECR 리포지토리와 AgentCore Runtime이 생성됩니다.

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인하겠습니다.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### 권한 부여 없이 AgentCore Runtime 호출

이제 payload로 AgentCore Runtime을 호출할 수 있습니다. 다음 셀을 실행하면 **"AccessDeniedException: An error occurred (AccessDeniedException) when calling the InvokeAgentRuntime operation: Agent is configured for a different authorization token type".** 오류가 표시됩니다.

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"})
invoke_response

### 권한 부여를 사용해 AgentCore Runtime 호출

올바른 권한 부여 토큰 유형으로 에이전트를 호출하겠습니다. 여기서는 Cognito access token을 사용합니다. "**Provision a Cognito User Pool**" 셀에서 access token을 복사하세요.

In [ ]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"))
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"}, bearer_token=bearer_token)
invoke_response

## 정리(선택 사항)

이제 생성한 AgentCore Runtime을 정리하겠습니다.

In [ ]:
from boto3.session import Session
import boto3

boto_session = Session()

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

# 축하합니다!